# SynBioCrow 2.2.0-rc1 — All Generators Setup
This notebook installs/imports SynBioCrow plus DORAnet, RetroBioCat2, RetroPath2/rp2paths, RetroTide, and (optionally) BioPKS Pipeline.\n\n**Important:** importable does not mean execution-ready. RBC2 needs its data assets; RetroPath2 needs KNIME + RetroRules + sink data; BioPKS remains an external authorized runtime and requires explicit license acknowledgement.\n

In [ ]:
import os, sys, subprocess, json, time, shutil, importlib, textwrap
from pathlib import Path
T0=time.time()
def status(p,msg): print(f'[SynBioCrow RC1] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {msg}',flush=True)
status(1,'environment')
print(sys.version)


## 1. Install SynBioCrow core + DORAnet + thermodynamics\nThis installs the consolidated branch and the two Python extras that are safe to install directly in the active notebook environment.\n

In [ ]:
repo=Path('/content/SynBioCrow')
RC_COMMIT='9f2af93fb79bf77c44e1c91ebc340b95f13234c0'
if repo.exists(): shutil.rmtree(repo)
status(5,f'fetch exact SynBioCrow RC integration commit {RC_COMMIT[:12]}')
repo.mkdir(parents=True,exist_ok=True)
subprocess.run(['git','-C',str(repo),'init','-q'],check=True)
subprocess.run(['git','-C',str(repo),'remote','add','origin','https://github.com/theiman112860/SynBioCrow.git'],check=True)
subprocess.run(['git','-C',str(repo),'fetch','-q','--depth','1','origin',RC_COMMIT],check=True)
subprocess.run(['git','-C',str(repo),'checkout','-q','--detach','FETCH_HEAD'],check=True)
resolved=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
assert resolved==RC_COMMIT,(resolved,RC_COMMIT)
print('[SYNBIOCROW PIN]',resolved,flush=True)
status(10,'install core + DORAnet + thermo')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)+'[doranet,thermo]'],check=True)


## 2. Install RetroBioCat2\nSynBioCrow uses RetroBioCat2 as a separate research runtime. This cell installs the proven upstream revision without letting it replace the whole scientific environment.\n

In [ ]:
status(20,'install RetroBioCat2 runtime dependencies')
subprocess.run([sys.executable,'-m','pip','install','-q','PyYAML','sqlalchemy','tables','gdown','tqdm'],check=True)
RBC2_REV='c5f32561b1ed99d0701a6dbb472dc448d6c082e8'
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--no-build-isolation',f'git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}'],check=True)
print('RetroBioCat2 installed. Real searches still require RBC2 scientific data assets.')


## 3. Install RetroPath2 wrapper + rp2paths\nThe Python packages can be installed here. Real RetroPath2 execution additionally requires KNIME, a RetroRules-ready rules file, and a sink file.\n

In [ ]:
status(32,'install RetroPath2 runtime dependencies')
subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'filetype','colored','colorlog','requests',
    'pydotplus','graphviz','lxml','numpy','pandas',
    'git+https://github.com/brsynth/brs-utils.git'
],check=True)
status(35,'install RetroPath2 Python components')
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/brsynth/retropath2-wrapper.git'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/brsynth/rp2paths.git'],check=True)
status(38,'smoke-test RetroPath2 imports before KNIME bootstrap')
mods=['brs_utils','filetype','colored','colorlog','requests','pydotplus','graphviz','lxml','retropath2_wrapper','rp2paths']
failed=[]
for name in mods:
    try:
        mod=importlib.import_module(name)
        print(f'[RETROPATH IMPORT] PASS {name} -> {getattr(mod,"__file__",None)}',flush=True)
    except Exception as exc:
        failed.append((name,type(exc).__name__,str(exc)))
        print(f'[RETROPATH IMPORT] FAIL {name}: {type(exc).__name__}: {exc}',flush=True)
if failed:
    raise RuntimeError('RetroPath Python import smoke failed: '+json.dumps(failed))
print('RETROPATH PYTHON IMPORT SMOKE PASS')
print('RetroPath2 Python components installed; KNIME/runtime resources will be configured below.')


## 4. Install RetroTide\nRetroTide is installed separately as a specialized PKS generator.\n

In [ ]:
status(50,'install RetroTide')
subprocess.run([sys.executable,'-m','pip','install','-q','mapchiral'],check=True)  # upstream RetroTide imports mapchiral but omits it from setup.py install_requires
subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/JBEI/RetroTide.git'],check=True)


## 5. BioPKS Pipeline in an isolated Python 3.10 environment
BioPKS Pipeline pins an older scientific stack (including DORAnet 0.5.6a1 and eQuilibrator <0.6), so it must **not** be installed into the main SynBioCrow environment. The upstream README recommends Python 3.10. This cell builds an isolated BioPKS runtime and keeps SynBioCrow connected through the external bridge contract.


In [ ]:
ACK_BIOPKS_LICENSE=True  # set True only after reviewing https://github.com/JBEI/BioPKS-Pipeline/blob/main/license.txt
BIOPKS_ROOT=Path('/content/BioPKS-Pipeline')
BIOPKS_VENV=Path('/content/biopks_py310')
BIOPKS_PYTHON=BIOPKS_VENV/'bin/python'

def run_live(cmd, *, timeout_s=900, label='subprocess'):
    import threading, queue
    q=queue.Queue()
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    def pump():
        for line in proc.stdout:
            q.put(line)
        q.put(None)
    threading.Thread(target=pump,daemon=True).start()
    start=time.time(); last=time.time(); done=False
    while True:
        try:
            item=q.get(timeout=1)
            if item is None:
                done=True
            else:
                print(item.rstrip(),flush=True); last=time.time()
        except queue.Empty:
            pass
        if done and proc.poll() is not None:
            break
        now=time.time()
        if now-last >= 30:
            print(f'[{label}] still running | elapsed {(now-start)/60:.1f} min',flush=True)
            last=now
        if now-start > timeout_s:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill()
            raise TimeoutError(f'{label} exceeded {timeout_s/60:.1f} min')
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode,cmd)
    return proc.returncode

BIOPKS_STATUS={'configured':False,'execution_ready':False,'reason':'license not acknowledged'}
if ACK_BIOPKS_LICENSE:
    status(62,'prepare isolated BioPKS Python 3.10 runtime')
    if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
    run_live(['git','clone','--depth','1','https://github.com/JBEI/BioPKS-Pipeline.git',str(BIOPKS_ROOT)],timeout_s=300,label='BioPKS clone')
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    run_live(['uv','python','install','3.10'],timeout_s=600,label='uv Python 3.10 install')
    if BIOPKS_VENV.exists(): shutil.rmtree(BIOPKS_VENV)
    run_live(['uv','venv','--python','3.10',str(BIOPKS_VENV)],timeout_s=300,label='BioPKS venv creation')

    # Upstream requirements contain obsolete `umap==0.1.1`, which is not present
    # in the package registry. BioPKS also already pins the actual package,
    # `umap_learn==0.5.2`. Patch only this temporary clone for Colab compatibility.
    upstream_req=BIOPKS_ROOT/'requirements.txt'
    patched_req=BIOPKS_ROOT/'requirements.synbiocrow_colab.txt'
    req_lines=upstream_req.read_text(encoding='utf-8').splitlines()
    removed=[line for line in req_lines if line.strip().lower()=='umap==0.1.1']
    kept=[line for line in req_lines if line.strip().lower()!='umap==0.1.1']
    patched_req.write_text('\n'.join(kept)+'\n',encoding='utf-8')
    print('[BIOPKS COMPAT PATCH] removed:',removed)
    print('[BIOPKS COMPAT PATCH] retained umap-learn pin:',[x for x in kept if 'umap' in x.lower()])

    run_live(['uv','pip','install','--python',str(BIOPKS_PYTHON),'-r',str(patched_req)],timeout_s=1200,label='BioPKS dependency install')
    run_live(['uv','pip','install','--python',str(BIOPKS_PYTHON),'--no-deps','-e',str(BIOPKS_ROOT)],timeout_s=600,label='BioPKS package install')
    os.environ['SYNBIOCROW_BIOPKS_ACK_LICENSE']='1'
    os.environ['SYNBIOCROW_BIOPKS_PYTHON']=str(BIOPKS_PYTHON)
    BIOPKS_STATUS={'configured':True,'execution_ready':BIOPKS_PYTHON.exists(),'python':str(BIOPKS_PYTHON),'compat_patch':'removed obsolete umap==0.1.1'}
else:
    print('BioPKS installation skipped. Review upstream license and set ACK_BIOPKS_LICENSE=True to install.')
print('[BIOPKS STATUS]',json.dumps(BIOPKS_STATUS,indent=2))


## 6. Import matrix\nThis checks Python importability independently from backend execution readiness.

In [ ]:
status(68,'restore SynBioCrow editable install after backend installs')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)],check=True)
if str(repo) not in sys.path:
    sys.path.insert(0,str(repo))
importlib.invalidate_caches()
spec=importlib.util.find_spec('synbiocrow')
print('[SYNBIOCROW SPEC]',spec)
subprocess.run([sys.executable,'-m','pip','show','synbiocrow'],check=False)
if spec is None:
    raise RuntimeError('SynBioCrow is still not importable after explicit reinstall + sys.path repair')

status(72,'import matrix')
MODULES={
 'synbiocrow':'synbiocrow',
 'DORAnet':'doranet',
 'RetroBioCat2':'rbc2',
 'RetroPath2 wrapper':'retropath2_wrapper',
 'rp2paths':'rp2paths',
 'RetroTide':'retrotide',
 'eQuilibrator':'equilibrator_api',
}
import_status={}
for label,name in MODULES.items():
    try:
        mod=importlib.import_module(name)
        import_status[label]={'importable':True,'module':getattr(mod,'__file__',None)}
    except Exception as exc:
        import_status[label]={'importable':False,'error':f'{type(exc).__name__}: {exc}'}

if ACK_BIOPKS_LICENSE and BIOPKS_PYTHON.exists():
    probe="import biopks_pipeline, json; print(json.dumps({'ok': True, 'module': getattr(biopks_pipeline, '__file__', None)}))"
    p=subprocess.run([str(BIOPKS_PYTHON),'-c',probe],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    import_status['BioPKS Pipeline']={'importable':p.returncode==0,'isolated_python':str(BIOPKS_PYTHON),'output':p.stdout[-2000:]}
else:
    import_status['BioPKS Pipeline']={'importable':False,'reason':'isolated runtime not installed'}
print(json.dumps(import_status,indent=2,sort_keys=True))


## 7. Configure and smoke-test external runtimes
This section makes RetroPath2 and BioPKS visible to SynBioCrow itself. RetroPath uses the upstream wrapper's supported KNIME bootstrap plus its published functional-test rules/sink corpus for validation only. BioPKS uses the real SynBioCrow JSON bridge in the isolated Python 3.10 environment.


In [ ]:
status(76,'configure BioPKS bridge and RetroPath runtime')

# --- BioPKS bridge ---
BIOPKS_BRIDGE=repo/'scripts'/'biopks_synbiocrow_bridge.py'
if ACK_BIOPKS_LICENSE and BIOPKS_PYTHON.exists():
    os.environ['SYNBIOCROW_BIOPKS_ACK_LICENSE']='1'
    os.environ['SYNBIOCROW_BIOPKS_PYTHON']=str(BIOPKS_PYTHON)
    os.environ['SYNBIOCROW_BIOPKS_ROOT']=str(BIOPKS_ROOT)
    os.environ['SYNBIOCROW_BIOPKS_RUNNER']=str(BIOPKS_BRIDGE)
    status(78,'smoke-test real SynBioCrow -> BioPKS bridge')
    run_live([sys.executable,str(repo/'scripts'/'smoke_biopks_real.py')],timeout_s=300,label='BioPKS bridge smoke')
else:
    print('[BIOPKS BRIDGE] skipped because BioPKS isolated runtime is not installed')

# --- RetroPath2 / KNIME / functional-test resources ---
RETROPATH_ROOT=Path('/content/retropath2_runtime')
# Persist the heavy KNIME archives in Google Drive so later Colab runs reuse them.
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
RETROPATH_CACHE=Path(os.environ.get(
    'SYNBIOCROW_RETROPATH_CACHE',
    '/content/drive/MyDrive/SynBioCrow/cache/knime46'
))
RETROPATH_CACHE.mkdir(parents=True,exist_ok=True)
print('[RETROPATH CACHE]',RETROPATH_CACHE,flush=True)
status(80,'bootstrap RetroPath2 KNIME 4.6.4 + targeted KNIME 4.6 p2 dependencies')
run_live([
    sys.executable,str(repo/'scripts'/'bootstrap_retropath2.py'),
    '--root',str(RETROPATH_ROOT),
    '--cache-dir',str(RETROPATH_CACHE),
    '--knime-version','4.6.4','--smoke-resources'
],timeout_s=7200,label='RetroPath bootstrap')
rp_manifest=json.loads((RETROPATH_ROOT/'synbiocrow_retropath_bootstrap.json').read_text())
os.environ['SYNBIOCROW_RETROPATH_RULES']=rp_manifest['rules_file']
os.environ['SYNBIOCROW_RETROPATH_SINK']=rp_manifest['sink_file']
os.environ['SYNBIOCROW_RETROPATH_KNIME']=rp_manifest['knime_install']
os.environ['SYNBIOCROW_RETROPATH_SMOKE_SOURCE']=rp_manifest['source_file']
status(84,'smoke-test SynBioCrow RetroPathBackend with upstream lycopene fixture')
run_live([sys.executable,str(repo/'scripts'/'smoke_retropath2_real.py')],timeout_s=1200,label='RetroPath smoke')
os.environ['SYNBIOCROW_RETROPATH_SMOKE_PASS']='1'


## 8. Unified SynBioCrow backend readiness
At this point readiness is reported by the engine itself using the same environment variables that real design runs consume. The RetroPath rules/sink used above are **upstream validation fixtures**, not a production resource choice.


In [ ]:
status(88,'SynBioCrow unified readiness')
from synbiocrow import SynBioCrowEngine
engine=SynBioCrowEngine()
readiness=engine.backend_readiness()
print(json.dumps(readiness,indent=2,sort_keys=True))
from synbiocrow.bootstrap import bootstrap_advice
for row in bootstrap_advice(engine):
    print(row)
print('RetroPath execution-ready:',readiness.get('retropath2',{}).get('execution_ready',False))
print('BioPKS isolated Python:',os.environ.get('SYNBIOCROW_BIOPKS_PYTHON'))
print('BioPKS bridge configured:',readiness.get('biopks_retrotide',{}).get('available',False))


## 9. Run M7/M9 core verification
This final step re-runs the release gates after the external-runtime integration smoke tests. It does not access protected benchmark truth.


In [ ]:
status(92,'run release-readiness gate')
subprocess.run([sys.executable,str(repo/'scripts/m7_release_readiness.py')],check=True)
status(97,'run M9 package check')
subprocess.run([sys.executable,str(repo/'scripts/build_m9_release.py'),'--check'],check=True)
status(100,'all-generators environment + bridge/runtime integration complete')
